# Cost-critic regularisers — Base vs L2 vs Dropout

**Standalone.** Data is pulled from wandb once and cached, so re-runs are offline.

Three ways of fitting the same CPO cost critic on `SafetyPointGoal1-v0`, each from its **own
sweep** and each pinned to one config rather than searched:

| method | sweep | what differs | critic lr |
|---|---|---|---|
| Base | `x1747wy1` | — | 3e-4 |
| L2 | `c20sedbr` | `critic-norm-coef-cost = 0.01` | 1e-4 |
| Dropout | `etqndcx7` | `critic.dropout-cost = 0.2` | 3e-4 |

All three share `algo=CPO`, `adv-estimation-method=plain`, `update-iters=2`, `cost-limit=10`
and seeds 1–5, so the columns below differ only in how the cost critic is regularised.

### The four columns

1. **Return vs cost** — final performance, one standard-error ellipse per method
2. **Correlation (reward)** — Corr($\hat V_r$, $\tilde V_r$) on the eval probes
3. **Correlation (cost)** — Corr($\hat V_c$, $\tilde V_c$) on the eval probes
4. **Estimation error (cost)** — $\hat V_c - \tilde V_c$; above zero the critic over-predicts

Columns 2–4 are all measured on the **eval probes, prediction against Monte-Carlo truth**.

### Layout

1. **Config** — sweeps, the pinned config per method
2. **Style** — `STYLE` and `TEXT`: every colour, size and string, in one place
3. **Utilities** — fetching, run selection, drawing primitives
4. **Load**
5. **The figure**

To restyle everything, edit `STYLE`/`TEXT` in section 2 and re-run.

## 1. Config

In [ ]:
import pathlib

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon

# Repo-anchored, so the same cache is used wherever the kernel starts.
REPO = pathlib.Path.cwd().resolve()
while not (REPO / "omnisafe").is_dir() and not (REPO / "plots").is_dir():
    if REPO.parent == REPO:
        raise RuntimeError("could not find the MICE repo root")
    REPO = REPO.parent

ENTITY_PROJECT = "liam-paull/calibration_rl"
DATA_DIR = REPO / "figs" / "critic_regularizers"
FIG_DIR = DATA_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Short name -> the wandb config key it comes from.  One table covers all three sweeps;
# a key a sweep never sets simply comes back NaN there.
CFG_COLUMNS = {
    "adv": "adv-estimation-method",
    "update_iters": "algo-cfgs.update-iters",
    "critic_lr": "model-cfgs.critic.lr",
    "cost_limit": "cost-limit",
    "l2_cost": "algo-cfgs.critic-norm-coef-cost",
    "dropout_cost": "model-cfgs.critic.dropout-cost",
}

SEEDS = [1, 2, 3, 4, 5]

# (name, sweep id, the pinned config).  Nothing is searched here: each method is one cell,
# given outright, and section 3.2 checks that exactly SEEDS runs come back.
# Order fixes plotting and legend order.
METHODS = [
    ("Base", "x1747wy1",
     dict(adv="plain", update_iters=2, cost_limit=10, critic_lr=0.0003)),
    ("L2", "c20sedbr",
     dict(adv="plain", update_iters=2, cost_limit=10, critic_lr=0.0001, l2_cost=0.01)),
    ("Dropout", "etqndcx7",
     dict(adv="plain", update_iters=2, cost_limit=10, critic_lr=0.0003, dropout_cost=0.2)),
]

COST_LIMIT = 10.0

# Which head column 4 shows.  Both are fetched, so switching needs no re-download.
ERROR_HEAD = "c"

HISTORY_KEYS = [
    "TotalEnvSteps", "Metrics/EpRet", "Metrics/EpCost",
    "PooledMC/Correlation_r", "PooledMC/Correlation_c",
    "PooledMC/EstimationError_c", "PooledMC/EstimationError_r",
]

print("repo   :", REPO)
print("cache  :", DATA_DIR)
print("methods:", ", ".join(n for n, _, _ in METHODS))

## 2. Style

`STYLE` is the palette, type, marks and panel geometry; `TEXT` is every string a figure
shows. Geometry is **inches per panel**, as in the other notebooks, so panels keep the same
physical size whatever the panel count.

Colour means **method** in every panel, using the Okabe-Ito hues the other notebooks use —
Base keeps the blue it has in `fig4.ipynb`. The three were checked pairwise: worst
colour-vision separation ΔE 11.0, worst normal-vision ΔE 18.7, all clearing 3:1 contrast.

This figure carries no y labels — each panel's title names its quantity — so `panel.left`
and `panel.wgap` only have to clear the tick labels.

In [ ]:
STYLE = {
    # colour means METHOD in every panel; Base keeps fig4's blue
    "method": {"Base": "#0072B2", "L2": "#009E73", "Dropout": "#D55E00"},
    "ink": {"title": "#1a1a1a", "label": "#5a5a5a", "spine": "#909090",
            "tick": "#8a8a8a", "grid": "#d8d8d8", "rule": "#5a5a5a"},

    "font": {"family": "DejaVu Sans", "title": 23, "label": 21, "tick": 17,
             "note": 16, "legend": 27, "suptitle": 28},

    "line": {"width": 2.4, "marker": 5, "band_alpha": 0.16,
             "grid": 1.0, "spine": 1.0, "tick_len": 3.6, "tick_width": 1.0,
             "rule_width": 1.7, "rule_dash": (0, (5, 3)),
             "ellipse_alpha": 0.20, "ellipse_edge": 2.0, "mean_marker": 13,
             "seed_marker": 6, "seed_alpha": 0.55},

    # inches.  wgap is tight -- there are no y labels to clear, only ticks.
    "panel": {"w": 4.30, "h": 4.30, "wgap": 0.62, "hgap": 1.45,
              "left": 1.00, "right": 0.12, "top": 0.62, "bottom": 1.05},

    "band": "sem",                      # "sem" or "sd"
    "dpi": 300, "pad_inches": 0.1,
    "xticks": [3e4, 1e5, 3e5, 1e6, 3e6, 1e7],
}

TEXT = {
    # First line is the context (which data, against what), second the quantity.
    "titles": ["Final performance\nReturn vs cost",
               "Eval, vs MC truth\nCorrelation (reward)",
               "Eval, vs MC truth\nCorrelation (cost)",
               "Eval, vs MC truth\nEstimation error (cost)"],
    "xlabel_steps": "Total env steps",
    "perf_x": "Avg episode cost",
    "cost_limit": "cost limit",
    "legend_title": None,
    "suptitle": "CPO cost critic: no regulariser vs L2 vs dropout",
}


def apply_style():
    """Push STYLE into rcParams. Re-run after editing STYLE."""
    f, l, ink = STYLE["font"], STYLE["line"], STYLE["ink"]
    mpl.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": STYLE["dpi"],
        "font.family": "sans-serif", "font.sans-serif": [f["family"]], "font.size": f["tick"],
        "axes.titlesize": f["title"], "axes.titlecolor": ink["title"], "axes.titlepad": 8,
        "axes.labelsize": f["label"], "axes.labelcolor": ink["label"],
        "axes.edgecolor": ink["spine"], "axes.linewidth": l["spine"],
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "axes.grid.axis": "y", "axes.axisbelow": True,
        "grid.color": ink["grid"], "grid.linewidth": l["grid"], "grid.alpha": 1.0,
        "xtick.labelsize": f["tick"], "ytick.labelsize": f["tick"],
        "xtick.color": ink["tick"], "ytick.color": ink["tick"],
        "xtick.labelcolor": ink["label"], "ytick.labelcolor": ink["label"],
        "xtick.direction": "out", "ytick.direction": "out",
        "xtick.major.size": l["tick_len"], "ytick.major.size": l["tick_len"],
        "xtick.major.width": l["tick_width"], "ytick.major.width": l["tick_width"],
        "xtick.minor.visible": False, "ytick.minor.visible": False,
        "legend.fontsize": f["legend"], "legend.frameon": False,
    })


apply_style()
MCOLOR = STYLE["method"]
print("style applied |", MCOLOR)

## 3. Utilities

Everything reusable, in one place: **3.1** fetching and caching, **3.2** run selection,
**3.3** drawing primitives.

### 3.1 Fetching and caching

In [ ]:
def save_csv(df, path):
    """Write via temp + rename, so an interrupted run leaves no half-written cache."""
    tmp = str(path) + ".tmp"
    df.to_csv(tmp, index=False)
    pathlib.Path(tmp).replace(path)


def to_numeric(df, cols):
    """Force numeric dtype.

    wandb's history() returns a sparsely-logged column (the PooledMC keys, present only at
    eval epochs) as dtype=object, while the same data via CSV comes back float64.
    Aggregating the object form concatenates the NaNs instead of averaging them.
    """
    for c in cols:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def fetch_summary(sweep_id):
    """One row per run of a sweep: the config columns in CFG_COLUMNS plus its summary."""
    import wandb
    api = wandb.Api(timeout=240)
    rows = []
    for r in api.sweep(f"{ENTITY_PROJECT}/{sweep_id}").runs:
        c = r.config
        d = {short: c.get(key) for short, key in CFG_COLUMNS.items()}
        d.update(run_id=r.id, state=r.state, seed=c.get("seed"),
                 EpRet=r.summary.get("Metrics/EpRet"),
                 EpLen=r.summary.get("Metrics/EpLen"),
                 TotalCost=r.summary.get("Metrics/TotalCost"),
                 TotalEnvSteps=r.summary.get("TotalEnvSteps"))
        rows.append(d)
    return pd.DataFrame(rows)


def fetch_history(run_ids):
    """Per-epoch curves, for the selected runs only -- not the whole sweep."""
    import wandb
    api = wandb.Api(timeout=240)
    out = []
    for i, rid in enumerate(run_ids, 1):
        h = api.run(f"{ENTITY_PROJECT}/{rid}").history(keys=HISTORY_KEYS, samples=100_000)
        h["run_id"] = rid
        out.append(h)
        print(f"    {i}/{len(run_ids)} {rid}", end="\r", flush=True)
    print()          # terminate the \r progress line, or the next print overwrites it
    return pd.concat(out, ignore_index=True)


def cached(path, fetch):
    """Read `path` if it exists, otherwise fetch and write it."""
    if path.exists():
        return pd.read_csv(path)
    df = fetch()
    save_csv(df, path)
    return df

### 3.2 Picking each method's runs

Nothing is scored here: every method's config is given outright in `METHODS`, so this only
has to *find* it. That makes a silent mismatch the danger — a key that never varied under
that name, or a value that was never run, would quietly return fewer seeds and shrink the
band rather than raise. So the match is checked against `SEEDS` and fails loudly.

In [ ]:
def select_runs(raw, match, seeds=SEEDS):
    """The finished runs matching a pinned config, verified to be exactly `seeds`."""
    d = raw[raw.state == "finished"].copy()
    m = np.ones(len(d), bool)
    for k, v in match.items():
        if k not in d.columns:
            raise KeyError(f"{k!r} is not a recorded config column; "
                           f"CFG_COLUMNS has {sorted(CFG_COLUMNS)}")
        col = d[k]
        if isinstance(v, str):
            m &= (col.astype(str) == v).to_numpy()
        else:
            m &= np.isclose(pd.to_numeric(col, errors="coerce").to_numpy(float), float(v))
    d = d[m]
    if seeds is not None:
        d = d[d.seed.isin(seeds)]
        got = sorted(int(s) for s in d.seed)
        if got != sorted(seeds):
            raise RuntimeError(f"config {match} matched seeds {got}, expected {sorted(seeds)}")
    return d.sort_values("seed").reset_index(drop=True)


def with_perf(d):
    """Add avg_ep_cost, from each run's own step count.

    A run that stopped short must not be credited with a lower total cost simply for having
    run less, so the divisor is its actual episode count rather than the nominal one.
    """
    d = d.copy()
    d["avg_ep_cost"] = d.TotalCost / (d.TotalEnvSteps / d.EpLen)
    return d

### 3.3 Drawing primitives

In [ ]:
def make_axes(ncols, nrows=1, legend_in=0.0, title_in=0.0, panel_w=None, panel_h=None):
    """Figure laid out from STYLE['panel'], in inches.

    legend_in reserves blank inches at the bottom for a common legend, title_in the same at
    the top for a figure title, so neither can land on the panels.
    """
    P = STYLE["panel"]
    w, h = panel_w or P["w"], panel_h or P["h"]
    left, bottom, top = P["left"], P["bottom"] + legend_in, P["top"] + title_in
    fig_w = left + ncols * w + (ncols - 1) * P["wgap"] + P["right"]
    fig_h = top + nrows * h + (nrows - 1) * P["hgap"] + bottom
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), squeeze=False)
    fig.subplots_adjust(left=left / fig_w, right=1 - P["right"] / fig_w,
                        top=1 - top / fig_h, bottom=bottom / fig_h,
                        wspace=P["wgap"] / w, hspace=P["hgap"] / h)
    return fig, (axes[0] if nrows == 1 else axes)


def fmt_steps(v, _=None):
    return f"{v/1e6:g}M" if v >= 1e6 else f"{v/1e3:g}k"


def log_steps(ax, x_first, x_last):
    """Shared log-steps x axis: fixed ticks labelled 30k ... 10M, no left margin."""
    ax.set_xscale("log")
    ax.set_xticks(STYLE["xticks"])
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_steps))
    ax.xaxis.set_minor_locator(mticker.NullLocator())
    ax.set_xlim(x_first, x_last)
    ax.set_xlabel(TEXT["xlabel_steps"])


def spread(M, how=None):
    """Seed mean and band half-width: standard error (default) or standard deviation."""
    M = np.asarray(M, float)
    m = np.nanmean(M, 0)
    sd = np.nanstd(M, 0, ddof=1) if M.shape[0] > 1 else np.zeros(M.shape[1])
    if (how or STYLE["band"]) == "sem":
        return m, sd / np.sqrt(np.maximum(np.sum(np.isfinite(M), 0), 1))
    return m, sd


def band(ax, x, M, color, lw=None, ls="-", marker=None, zorder=3):
    """Seed-mean line with its band."""
    L = STYLE["line"]
    m, e = spread(M)
    ax.plot(x, m, color=color, lw=lw or L["width"], ls=ls, zorder=zorder,
            marker=marker, markersize=L["marker"], markeredgewidth=0)
    ax.fill_between(x, m - e, m + e, color=color, alpha=L["band_alpha"], lw=0,
                    zorder=zorder - 1)
    return m


def sem_ellipse(ax, x, y, color, n_pts=180):
    """Standard-error ellipse of the mean of the (x, y) seed points.

    The covariance of the points describes seed-to-seed spread; dividing it by n turns that
    into the sampling covariance of the mean, so this is a 1-s.e. region for where the
    method sits -- not a region containing the seeds.

    Drawn as a sampled polygon rather than a patches.Ellipse because column 1's x axis is
    logarithmic: an Ellipse is placed by centre/width/height in data units and would render
    wrong under a non-affine scale, whereas boundary points are transformed individually.
    The statistics are computed in linear space either way.
    """
    n = len(x)
    if n < 3:
        return
    cov = np.cov(x, y) / n
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    if not np.all(np.isfinite(vals)) or (vals <= 0).any():
        return
    t = np.linspace(0, 2 * np.pi, n_pts)
    pts = vecs @ np.vstack([np.sqrt(vals[0]) * np.cos(t), np.sqrt(vals[1]) * np.sin(t)])
    L = STYLE["line"]
    ax.add_patch(Polygon(np.column_stack([pts[0] + np.mean(x), pts[1] + np.mean(y)]),
                         closed=True, facecolor=color, edgecolor=color,
                         alpha=L["ellipse_alpha"], linewidth=L["ellipse_edge"], zorder=3))


def _reading_order(entries, ncol):
    """Reorder entries so a wrapped legend reads left-to-right, top-to-bottom."""
    n = len(entries)
    nrow = -(-n // ncol)
    return [entries[r * ncol + c] for c in range(ncol) for r in range(nrow)
            if r * ncol + c < n]


def legend_below(fig, entries, ncol=4, y=0.012, title=None):
    """One legend in the space reserved by make_axes(legend_in=...).

    A legend wider than the figure is never clipped -- savefig(bbox_inches="tight") grows
    the canvas to fit it, padding the panels with blank margin instead.  So measure it and
    wrap to fewer columns until it fits the axes band, then clear the x labels below.
    """
    sp = fig.subplotpars
    band_w = sp.right - sp.left
    inv = fig.transFigure.inverted()
    leg = None
    for n in range(max(1, min(ncol, len(entries))), 0, -1):
        if leg is not None:
            leg.remove()
        ordered = _reading_order(entries, n)
        leg = fig.legend([h for h, _ in ordered], [l for _, l in ordered],
                         loc="lower center", ncol=n, frameon=False, title=title,
                         fontsize=STYLE["font"]["legend"], bbox_to_anchor=(0.5, y),
                         title_fontsize=STYLE["font"]["legend"])
        if title:
            leg.get_title().set_color(STYLE["ink"]["title"])
            leg.get_title().set_fontweight("bold")
        fig.canvas.draw()
        box = leg.get_window_extent().transformed(inv)
        if box.width <= band_w:
            break
    floor = min(a.get_tightbbox().transformed(inv).y0 for a in fig.get_axes())
    over = box.y1 - (floor - 0.015)
    if over > 0:
        leg.set_bbox_to_anchor((0.5, y - over), transform=fig.transFigure)
    return leg


def suptitle(fig, text):
    """Figure title, placed clear of the panel titles.

    The panel titles run to two lines here, so their height is measured rather than derived
    from the font size: anything assuming one line lands on top of them.
    """
    fig.canvas.draw()
    inv = fig.transFigure.inverted()
    tops = [a.title.get_window_extent().transformed(inv).y1
            for a in fig.get_axes() if a.title.get_text()]
    y = (max(tops) if tops else fig.subplotpars.top) + 0.015
    return fig.suptitle(text, fontsize=STYLE["font"]["suptitle"],
                        color=STYLE["ink"]["title"], y=y, va="bottom")


def thin_tick_labels(fig, ax, axis="x", min_gap_px=8):
    """Hide tick labels that would collide with the previous one, keeping every mark."""
    fig.canvas.draw()
    labels = ax.get_xticklabels() if axis == "x" else ax.get_yticklabels()
    span = (lambda b: (b.x0, b.x1)) if axis == "x" else (lambda b: (b.y0, b.y1))
    shown = sorted((t for t in labels if t.get_text()),
                   key=lambda t: span(t.get_window_extent())[0])
    edge = None
    for t in shown:
        lo, hi = span(t.get_window_extent())
        if edge is not None and lo < edge + min_gap_px:
            t.set_visible(False)
        else:
            edge = hi


def line_key(color, lw=None, ls="-"):
    return Line2D([], [], color=color, lw=lw or STYLE["line"]["width"], ls=ls)


def dot_key(color):
    return Line2D([], [], color=color, marker="o", ls="none",
                  markersize=STYLE["line"]["marker"] + 4, markeredgewidth=0)


def save(fig, stem, formats=("png", "pdf"), pad=None):
    """Write the figure.  `pad` overrides STYLE's border, which a small figure cannot
    afford: 0.1in each side is 6% of a 3.4in-wide one."""
    for ext in formats:
        fig.savefig(FIG_DIR / f"{stem}.{ext}", dpi=STYLE["dpi"], bbox_inches="tight",
                    pad_inches=STYLE["pad_inches"] if pad is None else pad,
                    facecolor="white")
    print("saved:", stem, "/".join(formats))


def show(*stems, width=1500):
    try:
        from IPython.display import Image, display
    except ImportError:
        return
    for s in stems:
        display(Image(filename=str(FIG_DIR / f"{s}.png"), width=width))

## 4. Load

Per method: read (or fetch) its sweep summary, find its pinned config, then read (or fetch)
the per-epoch history for just those five runs. Everything is cached under
`figs/critic_regularizers/`, one file per sweep.

In [ ]:
RUNS, HIST = {}, {}
for name, sid, match in METHODS:
    raw = cached(DATA_DIR / f"summary_{sid}.csv", lambda s=sid: fetch_summary(s))
    raw = to_numeric(raw, ["update_iters", "critic_lr", "cost_limit", "l2_cost",
                           "dropout_cost", "seed", "EpRet", "EpLen", "TotalCost",
                           "TotalEnvSteps"])
    sel = with_perf(select_runs(raw, match))
    RUNS[name] = sel
    print(f"{name:8s} {sid}  {len(raw):3d} runs in sweep -> {len(sel)} selected  "
          f"| EpRet {sel.EpRet.mean():6.2f}  avg ep cost {sel.avg_ep_cost.mean():6.2f}")

    h = cached(DATA_DIR / f"history_{sid}.csv",
               lambda ids=list(sel.run_id): fetch_history(ids))
    missing = sorted(set(sel.run_id) - set(h.run_id))
    if missing:
        print(f"    cache is missing {len(missing)} selected runs; fetching them")
        h = pd.concat([h, fetch_history(missing)], ignore_index=True)
        save_csv(h, DATA_DIR / f"history_{sid}.csv")
    HIST[name] = to_numeric(h, HISTORY_KEYS)

X_FIRST = min(h.TotalEnvSteps.min() for h in HIST.values())
X_LAST = max(h.TotalEnvSteps.max() for h in HIST.values())
print(f"\nsteps {X_FIRST:,.0f} -> {X_LAST:,.0f}")

## 5. The figure

One row, four columns, all on a log x axis. Colour is the method throughout; there are no y
labels, so each title names its own quantity.

Column 1 is the only one not against env steps: final performance, one 1-s.e. ellipse per
method, with cost on a log axis and the limit marked. Left and up is better. Columns 2–4 are
all measured on the eval probes, prediction against Monte-Carlo truth.

In [ ]:
FIG = {
    "perf_xlim": None, "perf_ylim": None,   # None -> from the data
    "perf_xpad": 1.05,                      # margin each side of column 1's x range
    "corr_ylim": (0.0, 1.0),
    "err_ylim": None,
    "show_seeds": False,                    # per-seed dots behind the ellipses
    "smooth_corr": 7,                       # running mean, in eval checkpoints
    "smooth_err": 3,
    "legend_in": 1.05,
    "title_in": 1.20,
    "legend_ncol": 3,
    "share_corr": True,                     # put columns 2-3 on one y scale
}


def smooth(M, w):
    """Running mean along the last axis, edge-normalised."""
    if w <= 1:
        return M
    k = np.ones(w)
    num = np.apply_along_axis(lambda v: np.convolve(v, k, "same"), -1, M)
    return num / np.convolve(np.ones(M.shape[-1]), k, "same")


def seed_curves(name, col):
    """(steps, per-seed matrix) for one history column, NaNs dropped first.

    The PooledMC columns are logged only at eval epochs, so a run's column is mostly NaN;
    dropping those rows before aligning keeps the curve intact, where carrying them through
    any windowed operation would erase it.
    """
    ids = set(RUNS[name].run_id)
    h = HIST[name][HIST[name].run_id.isin(ids)].dropna(subset=[col, "TotalEnvSteps"])
    per = [g.sort_values("TotalEnvSteps") for _, g in h.groupby("run_id")]
    per = [g for g in per if len(g)]
    if not per:
        return np.array([]), np.zeros((0, 0))
    steps = np.array(sorted(set.intersection(*[set(g.TotalEnvSteps) for g in per])))
    M = np.stack([np.interp(steps, g.TotalEnvSteps, g[col]) for g in per])
    return steps, M


def figure_regularizers():
    fig, ax = make_axes(4, legend_in=FIG["legend_in"], title_in=FIG["title_in"])
    L, ink = STYLE["line"], STYLE["ink"]
    names = [n for n, _, _ in METHODS]

    # --- 1. final performance: one s.e. ellipse per method ----------------------------
    for name in names:
        d, c = RUNS[name], MCOLOR[name]
        x, y = d.avg_ep_cost.to_numpy(), d.EpRet.to_numpy()
        if FIG["show_seeds"]:
            ax[0].scatter(x, y, s=L["seed_marker"] ** 2, color=c, alpha=L["seed_alpha"],
                          edgecolors="none", zorder=4)
        sem_ellipse(ax[0], x, y, c)
        ax[0].scatter(x.mean(), y.mean(), s=L["mean_marker"] ** 2, color=c,
                      edgecolor="white", linewidth=1.5, zorder=5)
    ax[0].axvline(COST_LIMIT, color=ink["rule"], lw=L["rule_width"], ls=L["rule_dash"],
                  zorder=2)
    ax[0].annotate(TEXT["cost_limit"], xy=(COST_LIMIT, 0.97),
                   xycoords=ax[0].get_xaxis_transform(), xytext=(-9, 0),
                   textcoords="offset points", rotation=90, ha="center", va="top",
                   fontsize=STYLE["font"]["note"], style="italic", color=ink["label"])
    # Episode cost spans well under a decade, so the default decade locator would label a
    # single tick; step within the decade instead and let thin_tick_labels prune.
    ax[0].set_xscale("log")
    ax[0].xaxis.set_major_locator(mticker.LogLocator(base=10,
                                                     subs=np.arange(1.0, 10.0, 0.2)))
    ax[0].xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax[0].xaxis.set_minor_locator(mticker.NullLocator())
    xs = np.concatenate([RUNS[n].avg_ep_cost.to_numpy() for n in names])
    pad = FIG["perf_xpad"]
    ax[0].set_xlim(min(xs.min(), COST_LIMIT) / pad, xs.max() * pad)
    ax[0].set_xlabel(TEXT["perf_x"])
    ax[0].grid(True, axis="both", color=ink["grid"], lw=L["grid"])
    if FIG["perf_xlim"]:
        ax[0].set_xlim(*FIG["perf_xlim"])
    if FIG["perf_ylim"]:
        ax[0].set_ylim(*FIG["perf_ylim"])

    # --- 2-4. eval diagnostics, prediction against Monte-Carlo truth ------------------
    sc, se = FIG["smooth_corr"], FIG["smooth_err"]
    panels = [(ax[1], "PooledMC/Correlation_r", sc, 1),
              (ax[2], "PooledMC/Correlation_c", sc, 1),
              # logged as mean(true - pred); negate for (pred - true)
              (ax[3], f"PooledMC/EstimationError_{ERROR_HEAD}", se, -1)]
    for a, key, w, sign in panels:
        for name in names:
            steps, M = seed_curves(name, key)
            if not len(steps):
                continue
            band(a, steps, smooth(sign * M, w), MCOLOR[name])
        log_steps(a, X_FIRST, X_LAST)
    for a in ax[1:3]:
        if FIG["corr_ylim"]:
            a.set_ylim(*FIG["corr_ylim"])
    ax[3].axhline(0, color=ink["rule"], lw=1.4, zorder=2)
    if FIG["err_ylim"]:
        ax[3].set_ylim(*FIG["err_ylim"])

    if FIG["share_corr"]:
        lims = [a.get_ylim() for a in ax[1:3]]
        lo, hi = min(l for l, _ in lims), max(h for _, h in lims)
        for a in ax[1:3]:
            a.set_ylim(lo, hi)

    for a, t in zip(ax, TEXT["titles"]):
        a.set_title(t)
    for a in ax:
        thin_tick_labels(fig, a)

    legend_below(fig, [(line_key(MCOLOR[n]), n) for n in names],
                 ncol=FIG["legend_ncol"], title=TEXT["legend_title"])
    suptitle(fig, TEXT["suptitle"])
    return fig


save(figure_regularizers(), "fig5_critic_regularizers")
show("fig5_critic_regularizers")

## 5.1 Minimal version, at half a page width

Two panels only — performance and the cost-critic correlation — sized for a paper column
rather than for reading on screen.

The point of a fixed `width_in` is that the **native** size is what sets the apparent type
size: a PDF included at its own width renders 9pt type at 9pt, whereas one drawn 7in wide
and scaled to half that renders it at 4.5pt. So this cell lays the figure out to
`MINI["width_in"]` and shrinks every font and stroke to match, instead of scaling the large
figure down.

`mini_style()` swaps the smaller values into `STYLE` for the duration, so the shared
primitives (`band`, `sem_ellipse`, `legend_below`) size themselves down without needing a
second copy of each. The saved width is reported below, since `bbox_inches="tight"` trims to
the drawn content and so lands near, not exactly on, the target.

In [ ]:
from contextlib import contextmanager

MINI = {
    "width_in": 3.4,        # target native width: half a ~6.75in text column
    "panel_h": 1.45,        # inches
    # Margins are sized to what actually sits in them, nothing more: `left` holds the
    # y label plus its ticks, `wgap` only the second panel's tick labels, `bottom` the
    # x label plus its ticks.  Every tenth of an inch here comes straight off the plots.
    # `right` is not spare space: the last x tick label is centred on the right spine,
    # so half of it sits outside the axes and would otherwise push the saved width
    # past width_in once bbox_inches="tight" grew the canvas to fit it.
    "left": 0.34, "wgap": 0.26, "right": 0.12, "top": 0.20, "bottom": 0.32,
    "legend_in": 0.20,
    "fonts": {"title": 9.5, "label": 8.5, "tick": 7.5, "legend": 8.5, "note": 7.0},
    "line": {"width": 1.3, "band_alpha": 0.18, "marker": 2.0,
             "ellipse_alpha": 0.20, "ellipse_edge": 0.9, "mean_marker": 5.0,
             "seed_marker": 2.5, "seed_alpha": 0.55,
             "rule_width": 1.0, "rule_dash": (0, (4, 2.5)),
             "grid": 0.6, "spine": 0.7, "tick_len": 2.2, "tick_width": 0.7},
    "corr_ylim": None,      # None -> autoscale; a tiny panel cannot spare the empty half
    "pad_inches": 0.02,     # STYLE's 0.1in border is 6% of this figure's width
    "limit_label": True,    # an unlabelled reference rule is a puzzle, not a cue
    "perf_xpad": 1.05,
    "smooth_corr": 7,
    "titles": ["Performance", "Correlation (cost)"],
    "ylabel": "Episode return",
    "legend_ncol": 3,
}


def mini_rc():
    """rcParams that are not read through STYLE."""
    f, l = MINI["fonts"], MINI["line"]
    return {"axes.titlesize": f["title"], "axes.labelsize": f["label"],
            "xtick.labelsize": f["tick"], "ytick.labelsize": f["tick"],
            "legend.fontsize": f["legend"], "axes.titlepad": 3.0,
            "axes.labelpad": 2.0, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5,
            "axes.linewidth": l["spine"], "grid.linewidth": l["grid"],
            "xtick.major.size": l["tick_len"], "ytick.major.size": l["tick_len"],
            "xtick.major.width": l["tick_width"], "ytick.major.width": l["tick_width"]}


@contextmanager
def mini_style():
    """Swap STYLE's type and marks for the small-figure set, restoring them on exit.

    The drawing primitives read their sizes out of STYLE, so overriding it here is what lets
    them be reused at this scale rather than duplicated.
    """
    saved_line, saved_font = dict(STYLE["line"]), dict(STYLE["font"])
    STYLE["line"].update(MINI["line"])
    STYLE["font"].update(MINI["fonts"])
    try:
        with mpl.rc_context(mini_rc()):
            yield
    finally:
        STYLE["line"].clear(); STYLE["line"].update(saved_line)
        STYLE["font"].clear(); STYLE["font"].update(saved_font)


def figure_minimal():
    M, names = MINI, [n for n, _, _ in METHODS]
    with mini_style():
        L, ink = STYLE["line"], STYLE["ink"]
        panel_w = (M["width_in"] - M["left"] - M["wgap"] - M["right"]) / 2
        fig_h = M["top"] + M["panel_h"] + M["bottom"] + M["legend_in"]
        fig, axes = plt.subplots(1, 2, figsize=(M["width_in"], fig_h), squeeze=False)
        ax = axes[0]
        fig.subplots_adjust(left=M["left"] / M["width_in"],
                            right=1 - M["right"] / M["width_in"],
                            top=1 - M["top"] / fig_h,
                            bottom=(M["bottom"] + M["legend_in"]) / fig_h,
                            wspace=M["wgap"] / panel_w)

        # --- performance ---------------------------------------------------------
        for name in names:
            d, c = RUNS[name], MCOLOR[name]
            x, y = d.avg_ep_cost.to_numpy(), d.EpRet.to_numpy()
            sem_ellipse(ax[0], x, y, c)
            ax[0].scatter(x.mean(), y.mean(), s=L["mean_marker"] ** 2, color=c,
                          edgecolor="white", linewidth=0.8, zorder=5)
        ax[0].axvline(COST_LIMIT, color=ink["rule"], lw=L["rule_width"],
                      ls=L["rule_dash"], zorder=2)
        if M["limit_label"]:
            ax[0].annotate(TEXT["cost_limit"], xy=(COST_LIMIT, 0.97),
                           xycoords=ax[0].get_xaxis_transform(), xytext=(-4, 0),
                           textcoords="offset points", rotation=90, ha="center",
                           va="top", fontsize=STYLE["font"]["note"], style="italic",
                           color=ink["label"])
        ax[0].set_xscale("log")
        ax[0].xaxis.set_major_locator(mticker.LogLocator(base=10,
                                                         subs=np.arange(1.0, 10.0, 0.5)))
        ax[0].xaxis.set_major_formatter(mticker.ScalarFormatter())
        ax[0].xaxis.set_minor_locator(mticker.NullLocator())
        xs = np.concatenate([RUNS[n].avg_ep_cost.to_numpy() for n in names])
        ax[0].set_xlim(min(xs.min(), COST_LIMIT) / M["perf_xpad"],
                       xs.max() * M["perf_xpad"])
        ax[0].set_xlabel(TEXT["perf_x"])
        ax[0].set_ylabel(M["ylabel"])
        ax[0].grid(True, axis="both", color=ink["grid"], lw=L["grid"])

        # --- correlation (cost) --------------------------------------------------
        for name in names:
            steps, Mc = seed_curves(name, "PooledMC/Correlation_c")
            if len(steps):
                band(ax[1], steps, smooth(Mc, M["smooth_corr"]), MCOLOR[name])
        log_steps(ax[1], X_FIRST, X_LAST)
        if M["corr_ylim"]:
            ax[1].set_ylim(*M["corr_ylim"])

        for a, t in zip(ax, M["titles"]):
            a.set_title(t)
        for a in ax:
            thin_tick_labels(fig, a)
        legend_below(fig, [(line_key(MCOLOR[n]), n) for n in names],
                     ncol=M["legend_ncol"])
        return fig


save(figure_minimal(), "fig5_minimal", pad=MINI["pad_inches"])

# bbox_inches="tight" trims to the drawn content, so report what actually landed on disk.
try:
    from PIL import Image as _PILImage
    with _PILImage.open(FIG_DIR / "fig5_minimal.png") as _im:
        _w, _h = (s / STYLE["dpi"] for s in _im.size)
    print(f"saved size: {_w:.2f} x {_h:.2f} in  (target width {MINI['width_in']:.2f} in)")
except ImportError:
    pass
show("fig5_minimal", width=760)

## 6. The runs behind each method

The config is pinned rather than searched, so this is the audit trail: which five runs each
curve is built from, and what they scored.

In [ ]:
rows = []
for name, sid, match in METHODS:
    for _, r in RUNS[name].iterrows():
        rows.append(dict(method=name, sweep=sid, seed=int(r.seed), run_id=r.run_id,
                         adv=r.adv, iters=int(r.update_iters), critic_lr=r.critic_lr,
                         l2_cost=r.l2_cost, dropout_cost=r.dropout_cost,
                         EpRet=round(float(r.EpRet), 2),
                         avg_ep_cost=round(float(r.avg_ep_cost), 2)))
table = pd.DataFrame(rows)
try:
    display(table)
except NameError:
    print(table.to_string(index=False))

## 7. Output

In [ ]:
for p in sorted(FIG_DIR.iterdir()):
    print(f"{p.stat().st_size / 1e3:8.1f} kB  {p.name}")